In [1]:
import pandas as pd

df = pd.read_csv("fire_archive_SV-C2_808949.csv")
print(df.shape)
print(df.columns.tolist())
df.head()

(3141953, 15)
['latitude', 'longitude', 'brightness', 'scan', 'track', 'acq_date', 'acq_time', 'satellite', 'instrument', 'confidence', 'version', 'bright_t31', 'frp', 'daynight', 'type']


,latitude,longitude,brightness,scan,track,acq_date,acq_time,satellite,instrument,confidence,version,bright_t31,frp,daynight,type
0,13.35601,79.31965,341.27,0.50,0.49,2021-01-01,731,SNPP,SNPP,n,2,294.72,5.14,D,0
1,13.08654,77.40105,333.95,0.34,0.56,2021-01-01,731,SNPP,SNPP,n,2,291.94,3.96,D,0
2,16.23978,77.88901,340.72,0.55,0.51,2021-01-01,732,SNPP,SNPP,n,2,298.59,6.50,D,0
3,16.26327,77.88449,333.44,0.55,0.51,2021-01-01,732,SNPP,SNPP,n,2,298.54,4.74,D,0
4,16.57349,79.47806,336.34,0.43,0.46,2021-01-01,732,SNPP,SNPP,n,2,296.89,1.79,D,0


In [2]:
print(df["acq_date"].min(), df["acq_date"].max())
print(df["confidence"].value_counts())

2021-01-01 2025-12-31
confidence
n    2509031
l     545468
h      87454
Name: count, dtype: int64


In [3]:
print(df["type"].value_counts())
print()
print((df["type"].value_counts(normalize=True) * 100).round(3))

type
0    2788417
2     348011
3       5490
1         35
Name: count, dtype: int64

type
0    88.748
2    11.076
3     0.175
1     0.001
Name: proportion, dtype: float64


In [4]:
static = df[df["type"] == 2]
print(len(static))
static[["latitude", "longitude", "acq_date", "frp", "confidence"]].head(10)

348011


,latitude,longitude,acq_date,frp,confidence
10,17.59905,83.18343,2021-01-01,14.20,n
35,19.96180,79.33607,2021-01-01,3.39,n
36,20.96020,85.18330,2021-01-01,7.48,n
37,20.95966,85.17957,2021-01-01,2.82,n
38,20.96298,85.17902,2021-01-01,2.82,l
39,20.96244,85.17529,2021-01-01,2.82,n
40,20.96189,85.17158,2021-01-01,1.92,n
41,20.96135,85.16785,2021-01-01,1.92,n
42,20.96522,85.17102,2021-01-01,2.99,n
43,20.96468,85.16730,2021-01-01,2.99,n


In [5]:
static = df[df["type"] == 2].copy()

# Round coordinates to 2 decimals (~1 km grid) to group repeat detections
static["site"] = static["latitude"].round(2).astype(str) + "_" + static["longitude"].round(2).astype(str)

sites = static.groupby("site").agg(
    detections=("site", "size"),
    days_detected=("acq_date", "nunique"),
    mean_frp=("frp", "mean"),
)

print("Unique ~1 km cells:", len(sites))
print(sites["days_detected"].describe())
sites.sort_values("days_detected", ascending=False).head(10)

Unique ~1 km cells: 1363
count    1363.000000
mean      162.137197
std       259.047370
min         1.000000
25%         8.000000
50%        44.000000
75%       189.500000
max      1305.000000
Name: days_detected, dtype: float64


,detections,days_detected,mean_frp
site,,,
23.78_86.39,5749,1305,3.005606
21.92_83.35,4866,1289,3.382478
21.11_72.65,2350,1288,6.926506
23.17_82.34,3656,1236,2.849584
20.96_85.17,3824,1223,3.006961
23.69_86.39,3326,1212,2.880583
23.76_86.4,4834,1206,2.949692
23.56_87.24,2902,1205,3.214652
23.8_86.33,2349,1193,2.937978


In [9]:
import folium

sites = static.groupby("site").agg(
    lat=("latitude", "mean"),
    lon=("longitude", "mean"),
    detections=("site", "size"),
    days_detected=("acq_date", "nunique"),
    mean_frp=("frp", "mean"),
).reset_index()

m = folium.Map(location=[22.5, 80], zoom_start=5, tiles=None)

folium.TileLayer(
    tiles="https://server.arcgisonline.com/ArcGIS/rest/services/World_Street_Map/MapServer/tile/{z}/{y}/{x}",
    attr="Esri",
    name="Streets",
).add_to(m)

folium.TileLayer(
    tiles="https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}",
    attr="Esri",
    name="Satellite",
).add_to(m)

for _, r in sites.iterrows():
    folium.CircleMarker(
        location=[r["lat"], r["lon"]],
        radius=3 + r["days_detected"] / 200,
        color="red" if r["days_detected"] > 500 else "orange",
        fill=True,
        fill_opacity=0.6,
        tooltip=f"{r['site']} | days: {r['days_detected']} | FRP: {r['mean_frp']:.1f}",
    ).add_to(m)

folium.LayerControl().add_to(m)

m.save("sites_map.html")

import os
os.startfile("sites_map.html")

In [10]:
import osmnx as ox

lat, lon = 27.67, 76.13   

tags = {
    "landuse": ["industrial", "quarry"],
    "man_made": ["works", "chimney", "flare"],
    "power": ["plant", "generator"],
}

osm = ox.features_from_point((lat, lon), tags=tags, dist=3000)
print(len(osm))
print(osm[["geometry"]].geom_type.value_counts())
osm.head()

8
Polygon         7
MultiPolygon    1
Name: count, dtype: int64


geometry  \
element  id                                                             
relation 17460325   MULTIPOLYGON (((76.13971 27.67104, 76.13971 27...   
way      561436583  POLYGON ((76.10299 27.65913, 76.10014 27.65668...   
         561436585  POLYGON ((76.108 27.66844, 76.10775 27.66774, ...   
         561436587  POLYGON ((76.10212 27.66517, 76.09835 27.66285...   
         561436589  POLYGON ((76.12633 27.67572, 76.12764 27.67522...   

                       landuse    mineral industrial barrier description  \
element  id                                                                
relation 17460325          NaN        NaN        NaN     NaN         NaN   
way      561436583  industrial        NaN        NaN     NaN         NaN   
         561436585      quarry        NaN        NaN     NaN         NaN   
         561436587      quarry        NaN        NaN     NaN         NaN   
         561436589      quarry  limestone        NaN     NaN         NaN   

                   plant:method plant:output:electricity plant:source  \
element  id                                                             
relation 17460325           NaN                      NaN          NaN   
way      561436583          NaN                      NaN          NaN   
         561436585          NaN                      NaN          NaN   
         561436587          NaN                      NaN          NaN   
         561436589          NaN                      NaN          NaN   

                        power          type generator:method  \
element  id                                                    
relation 17460325   generator  multipolygon     photovoltaic   
way      561436583        NaN           NaN              NaN   
         561436585        NaN           NaN              NaN   
         561436587        NaN           NaN              NaN   
         561436589        NaN           NaN              NaN   

                   generator:output:electricity generator:source  \
element  id                                                        
relation 17460325                           yes            solar   
way      561436583                          NaN              NaN   
         561436585                          NaN              NaN   
         561436587                          NaN              NaN   
         561436589                          NaN              NaN   

                              generator:type  
element  id                                   
relation 17460325   solar_photovoltaic_panel  
way      561436583                       NaN  
         561436585                       NaN  
         561436587                       NaN  
         561436589                       NaN

In [11]:
import geopandas as gpd
from shapely.geometry import Point

utm = osm.estimate_utm_crs()
osm_m = osm.to_crs(utm)
pt_m = gpd.GeoSeries([Point(lon, lat)], crs="EPSG:4326").to_crs(utm).iloc[0]

osm_m["kind"] = osm_m["landuse"].fillna(osm_m["power"])
osm_m["dist_m"] = osm_m.distance(pt_m).round(0)

print(osm_m[["kind", "mineral", "dist_m"]].sort_values("dist_m"))

                           kind    mineral  dist_m
element  id                                       
way      561436592   industrial        NaN     0.0
         561436589       quarry  limestone   369.0
         1271813121  industrial        NaN   909.0
relation 17460325     generator        NaN   927.0
way      561436585       quarry        NaN  1671.0
         561436587       quarry        NaN  2708.0
         561436583   industrial        NaN  2753.0
         662692744       quarry        NaN  3291.0


In [13]:
import time
import pandas as pd
import geopandas as gpd
import osmnx as ox
from shapely.geometry import Point

ox.settings.requests_timeout = 60   # give up on a single request after 60 s
ox.settings.log_console = False

tags = {
    "landuse": ["industrial", "quarry"],
    "man_made": ["works", "chimney", "flare"],
    "power": ["plant", "generator"],
}

def kind_of(row):
    for col in ["landuse", "power", "man_made"]:
        if col in row.index and pd.notna(row[col]):
            return f"{col}={row[col]}"
    return "other"

top = sites.sort_values("days_detected", ascending=False).head(5)
results = []

for i, (_, r) in enumerate(top.iterrows(), 1):
    t0 = time.time()
    out = {"site": r["site"], "days": r["days_detected"], "n_features": 0,
           "nearest_kind": None, "nearest_dist_m": None}
    try:
        feats = ox.features_from_point((r["lat"], r["lon"]), tags=tags, dist=1000)
        utm = feats.estimate_utm_crs()
        feats_m = feats.to_crs(utm)
        pt = gpd.GeoSeries([Point(r["lon"], r["lat"])], crs="EPSG:4326").to_crs(utm).iloc[0]
        feats_m["kind"] = feats_m.apply(kind_of, axis=1)
        feats_m["dist_m"] = feats_m.distance(pt)
        nearest = feats_m.sort_values("dist_m").iloc[0]
        out.update(n_features=len(feats), nearest_kind=nearest["kind"],
                   nearest_dist_m=round(nearest["dist_m"]))
    except Exception as e:
        out["error"] = str(e)[:60]
    results.append(out)
    print(f"{i}/5 done: {r['site']} in {time.time() - t0:.0f}s")
    time.sleep(1)

pd.DataFrame(results)

1/5 done: 23.78_86.39 in 71s
2/5 done: 21.92_83.35 in 10s
3/5 done: 21.11_72.65 in 22s
4/5 done: 23.17_82.34 in 24s
5/5 done: 20.96_85.17 in 13s


,site,days,n_features,nearest_kind,nearest_dist_m
0,23.78_86.39,1305,2,landuse=quarry,0
1,21.92_83.35,1289,5,landuse=industrial,0
2,21.11_72.65,1288,11,landuse=industrial,0
3,23.17_82.34,1236,1,landuse=quarry,17
4,20.96_85.17,1223,3,landuse=quarry,0


In [14]:
import os
os.makedirs("outputs", exist_ok=True)
sites.to_csv("outputs/static_sites.csv", index=False)
pd.DataFrame(results).to_csv("outputs/top5_osm_check.csv", index=False)